# Week 09 - 하이퍼파라미터 튜닝 실습
**빅데이터 응용 실습 | Hyperparameter Tuning Practice**

---

### 📌 학습 목표
- GridSearchCV를 활용한 하이퍼파라미터 튜닝
- RandomizedSearchCV를 활용한 하이퍼파라미터 튜닝
- 최적 모델 성능 비교 및 평가

## 1. 라이브러리 임포트

In [9]:
import numpy as np
import pandas as pd
import time

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import classification_report, f1_score

pd.set_option('display.max_columns', 200)
print('Ready')

Ready


1) 데이터 처리 라이브러리
- numpy: 수치 연산 및 배열 처리
- pandas: 데이터프레임 조작 및 분석
- time: 코드 실행 시간 측정

2) 모델 및 전처리 라이브러리
- sklearn: 머신러닝 모델, 교차검증
- StandardScaler: 특성 스케일링
- Pipeline: 전처리와 모델 통합

3) 평가 지표 및 설정 완료
- classification_report: 성능 보고서
- f1_score: F1 점수 계산

In [10]:
df = pd.read_csv('week9_hyperparameter_tuning_practice.csv')
x = df.drop(columns=['churn'])
y = df['churn']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.3, random_state = 42, stratify = y
)

print(y_train.value_counts(normalize=True).round(3))
print(y_test.value_counts(normalize=True).round(3))

churn
0    0.75
1    0.25
Name: proportion, dtype: float64
churn
0    0.75
1    0.25
Name: proportion, dtype: float64


In [11]:
# 튜닝 전

baseline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

baseline.fit(x_train, y_train)
pred_base = baseline.predict(x_test)

print(classification_report(y_test, pred_base, digits=3))

              precision    recall  f1-score   support

           0      0.920     0.941     0.930       135
           1      0.810     0.756     0.782        45

    accuracy                          0.894       180
   macro avg      0.865     0.848     0.856       180
weighted avg      0.893     0.894     0.893       180



1) 왜 기준 모델이 필요할까
- 튜닝 전 성능 기준점을 설정하여 튜닝 후 개선도를 정량적으로 비교.
- Logistic Regression은 해석력이 높음

2) StandardScaler가 하는 일
- 특성 표준화: 모든 특성을 평균 0, 표준편차 1로 변환하여 모델 수렴을 빠르게 하고 수치 안정성을 확보함

3) LogisticRegression(max_iter=1000)
- max_iter = 1000: 최대 반복 횟수를 1000으로 늘려 수렴 보장 및 학습의 안정성을 확보함

4) classification_report 해석
- precision: 예측 이탈 중 실제 비율
- recall: 실제 이탈 중 예측 비율
- f1-score: precision과 recall의 조화평균

---

전체 정확도는 89.4%로 양호하지만, class 1의 recall이 낮아 일부 양성 데이터를 놓치는 중
데이터 불균형으로 인해 모델이 클래스 0에 편향된 성향을 보임

churn 문제에서는 accuracy보다 recall과 f1이 중요하다.
이탈 고객을 놓치지 않는 것이 비즈니스 목표이므로 recall을 우선으로 한다.

In [18]:
#pipeline 없이 GridSearch 

scaler = StandardScaler()
x_train_scaled_bad = scaler.fit_transform(x_train)
x_test_scaled_bad = scaler.transform(x_test)

knn = KNeighborsClassifier()
param_grid = {'n_neighbors': [3, 5, 7, 9, 11, 15]}

start = time.time()
gs_bad = GridSearchCV(knn, param_grid, cv=5, scoring='f1')
gs_bad.fit(x_train_scaled_bad, y_train)
elapsed_bad = time.time() - start

print('Bad GridSearch (no Pipeline)')
print('best_params_:', gs_bad.best_params_)
print('best_score_ (cv):', round(gs_bad.best_score_, 4))
print('time:', round(elapsed_bad, 3))

#테스트 성능
pred_bed = gs_bad.best_estimator_.predict(x_test_scaled_bad)
print('\n[Test]:')
print(classification_report(y_test, pred_bed, digits=3))


Bad GridSearch (no Pipeline)
best_params_: {'n_neighbors': 5}
best_score_ (cv): 0.7261
time: 0.059

[Test]:
              precision    recall  f1-score   support

           0      0.919     0.919     0.919       135
           1      0.756     0.756     0.756        45

    accuracy                          0.878       180
   macro avg      0.837     0.837     0.837       180
weighted avg      0.878     0.878     0.878       180

